# 🥷 [심화] MNIST 모델 속이기: 적대적 예제

> ### 🗺️ 이 심화 실습에서 배우는 것
> 1. 내가 만든 숫자 판별기를 **일부러 속이는** 방법
> 2. **결정 경계**가 뭔지 눈으로 직접 확인하기
> 3. 작은 변화가 큰 결과를 만드는 수학 원리 (가중합과 소프트맥스)
>
> ⚠️ 선수 실습: `심화)_MNIST_0부터9까지_숫자판별.ipynb`를 먼저 해보세요!

In [ ]:
# 한글을 그래프에 보여주는 라이브러리
!pip install -q koreanize_matplotlib
import koreanize_matplotlib

## 1. 준비: 모델 다시 학습시키기

속이려면 먼저 모델이 있어야죠.
이번엔 빠르게 학습하려고 조금 작게 만듭니다.

In [ ]:
import numpy as np                      # 숫자 계산용
import matplotlib.pyplot as plt         # 그림 그리기용
from sklearn.datasets import fetch_openml            # MNIST 가져오기
from sklearn.neural_network import MLPClassifier     # 신경망 모델

plt.rcParams['axes.unicode_minus'] = False

# MNIST 불러오기
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data / 255.0                  # 정규화: 0~255 → 0~1
y = mnist.target.astype(int)            # 정답을 정수로

X_train, y_train = X[:60000], y[:60000]  # 학습용
X_test, y_test = X[60000:], y[60000:]    # 시험용

# 작은 모델로 빠르게 학습 (정확도는 조금 낮아도 실험엔 충분)
model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=10,
                      random_state=42, verbose=True)
model.fit(X_train, y_train)

print(f"\n🎯 시험 점수: {model.score(X_test, y_test)*100:.2f}%")

## 2. 실험 1: 숫자 섞기 — 3이 8로 변하는 순간

3 사진과 8 사진을 섞어봅시다.
섞는 비율을 조금씩 바꾸면서 모델의 답이 언제 바뀌는지 찾습니다.

```
섞은 사진 = (1-α) × 3사진 + α × 8사진
```

α=0이면 순수한 3, α=1이면 순수한 8입니다.

In [ ]:
# 3 사진과 8 사진 하나씩 고르기
idx3 = np.where(y_test == 3)[0][0]   # 시험용 중 첫 번째 3
idx8 = np.where(y_test == 8)[0][0]   # 시험용 중 첫 번째 8
img3 = X_test[idx3]                   # 3 사진 (784개 숫자)
img8 = X_test[idx8]                   # 8 사진 (784개 숫자)

print(f"3 사진의 모델 예측: {model.predict([img3])[0]}")
print(f"8 사진의 모델 예측: {model.predict([img8])[0]}")

# α를 0부터 1까지 11단계로 바꿔가며 섞기
alphas = np.linspace(0, 1, 11)        # [0, 0.1, 0.2, ..., 1.0]
fig, axes = plt.subplots(1, 11, figsize=(20, 3))

flip_alpha = None                      # 답이 바뀌는 지점 기록용
for i, a in enumerate(alphas):
    mixed = (1 - a) * img3 + a * img8  # 섞기!
    pred = model.predict([mixed])[0]   # 모델에게 물어보기
    proba = model.predict_proba([mixed])[0]  # 각 숫자일 확률

    axes[i].imshow(mixed.reshape(28, 28), cmap='gray')
    axes[i].set_title(f"α={a:.1f}\n예측:{pred}", fontsize=10,
                      color='red' if pred != 3 else 'green',
                      fontweight='bold')
    axes[i].axis('off')

    # 3이 아니게 되는 첫 순간 기록
    if pred != 3 and flip_alpha is None:
        flip_alpha = a

plt.suptitle("3 → 8 섞기 실험: 모델의 답이 바뀌는 순간은?", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n🔍 α={flip_alpha}에서 처음으로 3이 아니라고 답했습니다!")

## 3. 실험 2: 확률 그래프로 결정 경계 보기

섞는 비율에 따라 각 숫자일 확률이 어떻게 변하는지 그래프로 봅시다.
두 확률 곡선이 **교차하는 지점**이 바로 **결정 경계**입니다!

In [ ]:
# α에 따른 3일 확률과 8일 확률 추적하기
alphas_fine = np.linspace(0, 1, 51)   # 더 촘촘하게 51단계
prob_3 = []                            # 3일 확률들
prob_8 = []                            # 8일 확률들

for a in alphas_fine:
    mixed = (1 - a) * img3 + a * img8
    p = model.predict_proba([mixed])[0]  # 10개 숫자 확률
    prob_3.append(p[3])                   # 3일 확률만 꺼내기
    prob_8.append(p[8])                   # 8일 확률만 꺼내기

# 그래프 그리기
plt.figure(figsize=(10, 5))
plt.plot(alphas_fine, prob_3, 'g-', linewidth=2, label='3일 확률')
plt.plot(alphas_fine, prob_8, 'r-', linewidth=2, label='8일 확률')
plt.axhline(y=0.5, color='gray', linestyle='--', label='50% 기준선')
plt.xlabel('α (섞는 비율: 0=순수 3, 1=순수 8)', fontsize=12)
plt.ylabel('확률', fontsize=12)
plt.title('결정 경계: 두 곡선이 교차하는 지점', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 두 곡선이 만나는 곳 = 모델이 '3일까 8일까' 가장 헷갈리는 곳")
print("   이 지점을 결정 경계(Decision Boundary)라고 합니다!")

## 4. 실험 3: 내가 그린 그림으로 모델 속이기 🖌️

종이에 숫자를 **일부러 헷갈리게** 써서 사진 찍어 올리세요!
예: 3을 둥글게 써서 8처럼, 1을 삐딱하게 써서 7처럼

In [ ]:
from google.colab import files
import cv2

# 1. 내가 그린 숫자 사진 업로드
print("📤 숫자 그림 사진을 올려주세요!")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 2. 흑백으로 읽고 28×28로 줄이기 (MNIST와 같은 크기!)
img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)  # 흑백으로 읽기
img = cv2.resize(img, (28, 28))                    # 28×28로 크기 맞추기
img = img / 255.0                                  # 정규화

# 3. 모델에게 물어보기
pred = model.predict([img.reshape(784)])[0]        # 예측
proba = model.predict_proba([img.reshape(784)])[0] # 확률
top3 = np.argsort(proba)[-3:][::-1]                # 확률 높은 3개

# 4. 결과 보여주기
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title("내가 올린 그림", fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].barh([str(t) for t in top3], [proba[t] for t in top3])
axes[1].set_xlabel("확률", fontsize=12)
axes[1].set_title(f"모델의 답: {pred}", fontsize=14, fontweight='bold', color='red')
plt.tight_layout()
plt.show()

print(f"\n🥷 모델은 '{pred}'라고 답했습니다!")
print("   속이는 데 성공했나요? 😆")

## 🎓 정리: 왜 작은 변화에 속을까?

### 수학 원리 한 줄 요약

모델의 계산은 결국 **가중합**입니다:

```
점수(숫자 k) = w₁×픽셀₁ + w₂×픽셀₂ + ... + w₇₈₄×픽셀₇₈₄
```

1. 각 픽셀을 조금씩만 바꿔도 784개를 합치면 **점수가 크게** 바뀝니다
2. **소프트맥스**는 점수 차이를 확률 차이로 뻥튀기합니다
3. 그래서 사람 눈엔 똑같아 보여도 모델의 답은 뒤집힙니다

### 🤔 생각해보기
1. α 실험에서 결정 경계가 정확히 0.5가 아닌 이유는 뭘까요?
2. 3→8 말고 1→7이나 4→9로 섞으면 경계가 어디쯤 생길까요?
3. 이런 속임수가 자율주행차 표지판 인식에 쓰이면 어떤 일이 생길까요?

### 💡 다음 단계
- **방어**: 적대적 예제로도 학습시키면(model.fit에 섞어서) 더 튼튼해집니다
- **심화**: 그라디언트(미분)를 이용해 가장 효율적으로 속이는 픽셀을 찾는 방법이 있습니다